In [2]:
import torch
import torch.nn as nn
from torchdiffeq import odeint_adjoint as odeint
import matplotlib.pyplot as plt


class SynchronousMachine(nn.Module):
    def __init__(self, D, P_m, X_d_dash, H, time_tensor,
                 Rs, Re, X_q_dash, Xep, Tdp, Tqp, Efd, Xd, Xq):
        super(SynchronousMachine, self).__init__()
        self.D = D
        self.P_m = P_m
        self.omega_s0 = 0.0
        self.omega_B = 2 * torch.pi * 50
        self.X_d_dash = X_d_dash
        self.H = H
        self.Rs = Rs
        self.Re = Re
        self.X_q_dash = X_q_dash
        self.Xep = Xep
        Z = torch.tensor([[self.Rs, -self.X_q_dash], [self.X_d_dash, self.Rs]])
        self.Z_inverse = torch.linalg.inv(Z)
        self.E_fd = Efd
        self.T_d_dash = Tdp
        self.T_q_dash = Tqp
        self.X_d = Xd
        self.X_q = Xq
        self.time_tensor = time_tensor
        self.V_tensor = None
        self.theta_tensor = None

    def set_time_tensor(self, time_tensor):
        self.time_tensor = time_tensor

    def set_V_tensor(self, V_tensor):
        # V_tensor: shape (n_time, 2)
        self.V_tensor = V_tensor[:, 0]
        self.theta_tensor = V_tensor[:, 1]

    def get_V(self, t):
        if self.V_tensor is None or self.time_tensor is None:
            raise ValueError("Voltage data not set.")
        idx = int((torch.abs(self.time_tensor - t)).argmin())
        return self.V_tensor[idx]

    def get_theta(self, t):
        if self.theta_tensor is None or self.time_tensor is None:
            raise ValueError("Angle data not set.")
        idx = int((torch.abs(self.time_tensor - t)).argmin())
        return self.theta_tensor[idx]

    def forward(self, t, y):
        delta, omega, E_d_dash, E_q_dash = y
        V_t = self.get_V(t)
        theta_vs = self.get_theta(t)
        v1 = E_d_dash - V_t * torch.sin(delta - theta_vs)
        v2 = E_q_dash - V_t * torch.cos(delta - theta_vs)
        I_d = self.Z_inverse[0, 0] * v1 + self.Z_inverse[0, 1] * v2
        I_q = self.Z_inverse[1, 0] * v1 + self.Z_inverse[1, 1] * v2
        P_e = E_d_dash * I_d + E_q_dash * I_q + (self.X_q_dash - self.X_d_dash) * I_d * I_q
        ddelta_dt = omega
        domega_dt = (self.omega_B / (2 * self.H)) * (self.P_m - P_e - self.D * omega)
        # Internal voltages assumed constant
        dE_d_dash_dt = torch.tensor(0.0, dtype=y.dtype, device=y.device)
        dE_q_dash_dt = torch.tensor(0.0, dtype=y.dtype, device=y.device)
        return torch.stack([ddelta_dt, domega_dt, dE_d_dash_dt, dE_q_dash_dt])


# class Generator4D(nn.Module):
#     def __init__(self, params, time_points, init_points, device='cpu'):
#         super(Generator4D, self).__init__()
#         # init_points: [delta, omega, E_d_dash, E_q_dash, P_m]
#         self.SyncMachine = SynchronousMachine(
#             params['D'], init_points[-1], params['X_d_dash'], params['H'],
#             time_points.squeeze(1), params['Rs'], params['Re'], params['X_q_dash'],
#             params['Xep'], params['Tdp'], params['Tqp'], params['Ef'], params['Xd'], params['Xq']
#         )
#         self.init_state = init_points[:4]
#         self.time_points = time_points

#     def set_event(self, time, P_m):
#         self.SyncMachine.P_m = P_m
#         self.event_time = time

#     def forward(self, V, times):
#         V_profile = V[:, 0, :]
#         self.SyncMachine.set_V_tensor(V_profile)
#         self.SyncMachine.set_time_tensor(times.squeeze(1))
#         sol = odeint(self.SyncMachine, self.init_state, self.time_points.squeeze(1), method='rk4')
#         delta = sol[:, 0]
#         E_d_dash = sol[:, 2]
#         E_q_dash = sol[:, 3]
#         Vs = V_profile[:, 0]
#         theta_vs = V_profile[:, 1]
#         v1 = E_d_dash - Vs * torch.sin(delta - theta_vs)
#         v2 = E_q_dash - Vs * torch.cos(delta - theta_vs)
#         I_d = self.SyncMachine.Z_inverse[0, 0] * v1 + self.SyncMachine.Z_inverse[0, 1] * v2
#         I_q = self.SyncMachine.Z_inverse[1, 0] * v1 + self.SyncMachine.Z_inverse[1, 1] * v2
#         I_D = I_d * torch.cos(delta - torch.pi/2) - I_q * torch.sin(delta - torch.pi/2)
#         I_Q = I_d * torch.sin(delta - torch.pi/2) + I_q * torch.cos(delta - torch.pi/2)
#         return I_D, I_Q, torch.tensor(0.0)


# def main():
#     device = 'cpu'
#     params = {
#         'H': 23.64, 'D': 2.364, 'Xd': 0.146, 'X_d_dash': 0.0608,
#         'Xq': 0.0969, 'X_q_dash': 0.0969, 'Tdp': 8.96, 'Tqp': 0.31,
#         'Rs': 0.0, 'Pm': 0.71, 'Ef': 1.08, 'Xep': 0.0, 'Re': 0.0
#     }
#     # Initial state: [delta, omega, E_d_dash, E_q_dash, P_m]
#     init_points = torch.tensor([0.0680, 0.0, 0.0, 1.0153, 0.7195])
#     time_points = torch.linspace(0, 2, 100, device=device).unsqueeze(1)

#     gen = Generator4D(params, time_points, init_points, device)
    
#     # Now callable via __call__
#     V_mag = torch.ones(100, 1)
#     V_ang = torch.zeros(100, 1)
#     V_profile = torch.cat((V_mag, V_ang), dim=1).unsqueeze(1)  # shape (100,1,2)

#     I_D, I_Q, _ = gen(V_profile, time_points)

#     # Plot results
#     plt.figure(figsize=(8,4))
#     plt.plot(time_points.squeeze(), I_D.detach().numpy(), label='I_D')
#     plt.plot(time_points.squeeze(), I_Q.detach().numpy(), label='I_Q')
#     plt.title('Generator Currents over Time')
#     plt.xlabel('Time [s]')
#     plt.ylabel('Current [A]')
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

# if __name__ == "__main__":
#     main()


In [9]:
import torch
from torchdiffeq import odeint
from ..src.solver import solver, ODE_conventional  # your base classes
# from your_module import SynchronousMachine   # wherever you defined it

# 1) Create a time‐vector and a corresponding “voltage profile” for the machine.
#    Here we just hold Vs = 1.0∠0° constant.
t_final, num_points = 10.0, 1000
t_tensor = torch.linspace(0., t_final, num_points)
Vs      = torch.ones(num_points)      # pu magnitude
θs      = torch.zeros(num_points)     # pu angle
V_profile = torch.stack((Vs, θs), dim=1)  # shape (num_points, 2)

# 2) Instantiate the machine with your chosen parameters.
#    SynchronousMachine.forward has signature (t, y) → dy/dt,
#    but it needs access to V_profile and t_tensor internally.
params = {
    'D': 2.364, 'P_m': 0.71, 'X_d_dash': 0.0608, 'H': 23.64,
    'Rs': 0.0, 'Re': 0.0, 'X_q_dash': 0.0969, 'Xep': 0.0,
    'Tdp': 8.96, 'Tqp': 0.31, 'Efd': 1.08, 'Xd': 0.146, 'Xq': 0.0969
}
machine = SynchronousMachine(
    params['D'], params['P_m'], params['X_d_dash'], params['H'],
    t_tensor, params['Rs'], params['Re'], params['X_q_dash'], params['Xep'],
    params['Tdp'], params['Tqp'], params['Efd'], params['Xd'], params['Xq']
)
machine.set_time_tensor(t_tensor)
machine.set_V_tensor(V_profile.unsqueeze(1))  # (T,1,2) as your Generator4D expects

# 3) Wrap its `forward` as the ODE‐function for your solver.
#    No extra args needed since everything is on `machine` itself.
solver = ODE_conventional(machine.forward)

# 4) Pick an initial state [δ₀, ω₀, E′d₀, E′q₀]
x0 = [0.0680, 0.0, 0.0, 1.0153]

# 5) Solve!
t, sol = solver.solve(x0, t_final, num_points)

# 6) Extract and plot
δ = sol[:,0].detach().numpy()
ω = sol[:,1].detach().numpy()

import matplotlib.pyplot as plt
plt.plot(t.numpy(), δ, label='δ(t)')
plt.plot(t.numpy(), ω, label='ω(t)')
plt.legend()
plt.xlabel('Time [s]')
plt.show()


ImportError: attempted relative import with no known parent package

In [8]:
import sys
print("\n".join(sys.path))


/teamspace/studios/this_studio
/teamspace/studios/this_studio/PINNProof/PINNProof/examples
/home/zeus/miniconda3/envs/cloudspace/lib/python310.zip
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/lib-dynload

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages


In [ ]:
ic_ranges = {
  'delta': (-0.1, 0.1),
  'omega': (-0.2, 0.2),
  'E_d_dash': (0.0, 0.1),
  'E_q_dash': (0.9, 1.1),
}
t, data, ics = solver.generate_dataset(
    ic_ranges=ic_ranges,
    num_ic=100,
    t_final=t_final,
    num_points=num_points,
    sampling='lhs',
    save_path='datasets/sync_machine.npz'
)
# → data.shape == (100, 1000, 4)
